# 02 - Modeling

Can a model that sees only the cells around a candidate estimate its chance of being a mine as well as the exact oracle? This notebook trains baselines, gradient-boosted trees, and a convolutional network on the 5,000-game dataset and scores them against the oracle.

**Data split (by whole game, so rows from one game never land on both sides):**

| Set | Games | Used for |
| --- | --- | --- |
| `fit` | 3,500 | training |
| `val` | 500 | early stopping |
| `test` | 1,000 | all scores below |

Fitted models are saved to `results/` as `.pkl` files, and the results table as `results/model_results.csv`.

In [ ]:
%load_ext autoreload
%autoreload 2

import pickle
import time

import numpy as np
from numpy import concatenate as conc
import pandas as pd

from inf_minesweeper.simulate import Config, run
from inf_minesweeper.loader import load, split_by_game
from inf_minesweeper.models import (Constant,                   # baseline: constant mine density
                                    LocalRatio,                 # baseline: average no. of mines around candidate cell
                                    LogisticFeatures,           # baseline/model: logistic regression
                                    GradientBoosted,            # model: LightGBM (PyPI v4.7.0)
                                    ConvNeuralNet)              # model: Convolutional NN (PyTorch)
from inf_minesweeper.metrics import evaluate

d = load("../data/raw/guesses.npz")
W, y, q, game, pos = d["window"], d["mine"], d["q"], d["game"], d["position"]

train, test = split_by_game(game)                       # 4,000 / 1,000 games
# hold out some training games for validation
fit, val = split_by_game(np.where(train, game, -1), test_frac=0.125, seed=1)
fit, val = fit & train, val & train

baseline = Constant().fit(W[train], y[train]).predict(W[test])
results = {}


def score(name, p):
    results[name] = evaluate(y[test], p, q[test], game[test], pos[test], baseline)
    return pd.DataFrame(results).T.round(3)

## Baselines

Four reference points that need little or no learning:

- **Constant, density 0.21:** every cell gets the board's mine density. This ignores that candidates touching numbers are riskier than average.
- **Constant, mine rate:** every cell gets the training set's mine rate (about 0.42). The zero point for "gap closed".
- **Local ratio:** a player's quick estimate, reading each touching number on its own and averaging (mines it still needs ÷ unknown cells around it).
- **Logistic regression:** 8 handcrafted features, such as how many numbers touch the candidate and what they imply.

**How to read the tables:**

| Column | Meaning | Better |
| --- | --- | --- |
| `log_loss`, `brier` | Error against the true mine labels | lower |
| `excess` | Log-loss above the oracle's; 0 is perfect | lower |
| `ece` | Calibration error: does a predicted 30% happen 30% of the time? | lower |
| `gap_closed` | Share of the way from the mine-rate constant to the oracle | higher |
| `hit_rate` | Chance of a mine if the model picks the guess | lower |
| `best_rate` | The same for the oracle's own pick, the floor | |
| `regret` | `hit_rate` minus `best_rate` | lower |
| `agreement` | Share of positions where the model picks a cell as safe as the oracle's | higher |

In [ ]:
for name, model in {
    "constant, density 0.21": Constant(0.21),
    "constant, mine rate": Constant(),
    "local ratio": LocalRatio(),
    "logistic, 8 features": LogisticFeatures(),
}.items():
    p = model.fit(W[train], y[train]).predict(W[test])
    score(name, p)
pd.DataFrame(results).T.round(3)

## Gradient-boosted trees (LightGBM)

Inputs: the k×k window around the candidate, one-hot encoded (13 states per cell), plus the 8 handcrafted features. Trees are added until the validation loss stops improving for 100 rounds.

The question here is whether a wider window helps: 3×3 sees only the candidate's neighbors, 7×7 sees three cells out. For the trees, 5×5 is best and 7×7 adds nothing.

In [ ]:
trained = {}
for k in (3, 5, 7):
    start = time.time()
    model = GradientBoosted(k=k).fit(W[fit], y[fit], 
                                     val=(W[val], y[val]))
    score(f"LightGBM {k}x{k}", model.predict(W[test]))
    trained[f"LightGBM {k}x{k}"] = model
    with open(f"../results/pickles/lgb{k}.pkl", "wb") as f:
        pickle.dump(model, f)
    print(f"k = {k}: {model.n_trees} trees, lgb{k}.pkl saved ({(time.time() - start):.2f}s)")

pd.DataFrame(results).T.round(3)

## Convolutional neural network (PyTorch)

`layers` sets the window exactly: each unpadded 3×3 convolution trims one cell from every side, so `layers=1..4` sees 3×3, 5×5, 7×7, and 9×9 and ends on a single output for the candidate. Training uses a random rotation or reflection of every batch, since the rules are the same in all 8 orientations, and stops early on the validation loss.

"Symmetrized" rows average each prediction over all 8 orientations. Each model is saved twice: `cnn{k}.pkl` predicts plainly, and `cnn{k}_sym.pkl` symmetrizes by default, for use as a `--policy` in self-play.

**Result:** unlike the trees, the CNN keeps improving as the window grows, so there is useful information beyond two cells away. The 9×9 symmetrized model closes 69% of the gap to the oracle, against 48% for the best LightGBM model. It is the least calibrated, though (ECE 0.023).

In [ ]:
for layers in (1, 2, 3, 4):
    start = time.time()
    model = ConvNeuralNet(layers=layers).fit(W[fit], y[fit], val=(W[val], y[val]))
    name = f"CNN {model.k}x{model.k}"
    score(name, model.predict(W[test]))
    score(name + ", symmetrized", model.predict(W[test], symmetrize=True))
    trained[name] = model
    with open(f"../results/pickles/cnn{model.k}.pkl", "wb") as f:
        pickle.dump(model, f)
    print(f"{name}: best epoch {model.best_epoch} of {len(model.history)}, cnn{model.k}.pkl saved ({(time.time() - start):.2f}s)")

for k in (5, 7, 9):
    with open(f"../results/pickles/cnn{k}.pkl", "rb") as f:
        model = pickle.load(f)
    model.symmetrize = True
    with open(f"../results/pickles/cnn{k}_sym.pkl", "wb") as f:
        pickle.dump(model, f)

table = pd.DataFrame(results).T.round(3)
table.to_csv("../results/metrics.csv")
print("\n >> metrics.csv saved")
display(table)

## Density shift

The app's mine density is estimated at 0.21 $\pm$ 0.01. Here the trained models are scored, unchanged, on 1,000 games simulated at densities 0.20 and 0.22 on seeds that never appear in training. Positions at 0.22 have nearly twice as many candidates as at 0.20, so this also tests whether the models handle bigger, more tangled positions.

CNN rows use symmetrized predictions. Compare `excess` with each model’s row on the 0.21 test set above: it changes by at most 0.004 for any model, and calibration error stays under 0.02, so the models hold up across the uncertainty in the density.

In [ ]:
shift = {}
for p, path in [(0.20, "../data/raw/shift_p020.npz"), (0.22, "../data/raw/shift_p022.npz")]:
    s = load(path)
    base = np.full(len(s["q"]), y[train].mean())        # same constant as on the test set
    for name, model in trained.items():                 # dict of name -> fitted model
        p_hat = model.predict(s["window"])
        if isinstance(model, ConvNeuralNet):
            model.symmetrize = True
        shift[(name, p)] = evaluate(s["mine"], p_hat, s["q"], s["game"], s["position"], base)

table = pd.DataFrame(shift).T.round(3)
table.to_csv("../results/shift.csv")
print(" >> shift.csv saved")
display(table)

## Policy shift: DAgger experiment

In the cell above, we asked “does the model still work if the density estimate is a bit off?”, and the answer was yes. Now we ask, “can the self-play problem be fixed?”, which is answered by a DAgger (Dataset Aggregation) experiment. The models are retrained on their own games and checked if DAgger beats the control, which means the improvement comes from showing the model its own positions.

In [ ]:
def retrain_on(extra_paths, make_model):
    """Train a fresh model on the original fit/val games plus each extra
    dataset's games, split the same way (7/8 for fitting, 1/8 for early
    stopping). Each extra is a file path or a dataset already in memory."""
    Wf, yf, Wv, yv = [W[fit]], [y[fit]], [W[val]], [y[val]]
    for extra in extra_paths:
        e = load(extra) if isinstance(extra, str) else extra
        e_fit, e_val = split_by_game(e["game"], test_frac=0.125, seed=2)
        Wf.append(e["window"][e_fit]); yf.append(e["mine"][e_fit])
        Wv.append(e["window"][e_val]); yv.append(e["mine"][e_val])
    return make_model().fit(conc(Wf), conc(yf), val=(conc(Wv), conc(yv)))


experiments = [
    # name                      # subset        # path                      # model to build
    ("LightGBM 5x5, dagger",    "lgb5_dagg",    "played/lgb5.npz",          lambda: GradientBoosted(k=5)),
    ("LightGBM 5x5, control",   "lgb5_ctrl",    "played/oracle.npz",        lambda: GradientBoosted(k=5)),
    ("CNN 9x9, dagger",         "cnn9_dagg",    "played/cnn9_sym.npz",      lambda: ConvNeuralNet(layers=4)),
    ("CNN 9x9, control",        "cnn9_ctrl",    "played/oracle.npz",        lambda: ConvNeuralNet(layers=4)),
]

for name, file, path, make_model in experiments:
    model = retrain_on([f"../data/raw/{path}"], make_model)
    if isinstance(model, ConvNeuralNet):
        model.symmetrize=True
    score(name, model.predict(W[test]))
    with open(f"../results/pickles/{file}.pkl", "wb") as f:
        pickle.dump(model, f)

compare = ["LightGBM 5x5", "LightGBM 5x5, dagger", "LightGBM 5x5, control",
           "CNN 9x9, symmetrized", "CNN 9x9, dagger", "CNN 9x9, control"]
comparison = pd.DataFrame(results).T.loc[compare].round(3)
comparison.to_csv("../results/dagg_metrics.csv")
print(" >> dagg_metrics.csv saved")
display(comparison)

### Analyzing the results

All 8 guessers playing seeds `300_000` to `300_999`; comparisons made game-by-game with confidence intervals.

In [ ]:
names = ["random", "lgb5", "lgb5_dagg", "lgb5_ctrl",
         "cnn9_sym", "cnn9_dagg", "cnn9_ctrl", "oracle"]
runs = {n: load(f"../data/raw/eval/{n}.npz") for n in names}

def same_safest_cell(d):
    """Share of forced guesses whose safest cell is the same as the oracle."""
    key = d["game"] * 1000 + d["position"]
    best = {}
    for k, xx, yy, qq in zip(key, d["x"], d["y"], d["q"]):
        if k not in best or qq < best[k][0]:
            best[k] = (qq, xx, yy)
    ks = sorted(best)
    return np.mean([best[a][1:] == best[b][1:] for a, b in zip(ks, ks[1:]) if (a // 1000) == (b // 1000)])

rand, orac = runs["random"]["game_score"], runs["oracle"]["game_score"]
rows = {}
for n, d in runs.items():
    s = d["game_score"]
    gap = s - rand
    n_q = d["game_pick_q_n"].sum()
    rows[n] = {
        "score": s.mean(),
        "vs random": gap.mean(),
        "± 95%": 1.96 * gap.std(ddof=1) / np.sqrt(len(gap)),
        "share of oracle lead": gap.mean() / (orac - rand).mean(),
        "cleared": d["game_cleared"].mean(),
        "locked": d["game_locked"].mean(),
        "guess mine chance": d["game_pick_q_sum"].sum() / n_q,
        "regret": d["game_regret_sum"].sum() / n_q,
        "safest cell unchanged": same_safest_cell(d),
    }
    
selfplay = pd.DataFrame(rows).T.round(3)
selfplay.to_csv("../results/selfplay.csv")
print(" >> selfplay.csv saved")
display(selfplay)

DAgger did not help after one round of imitation.

### Iterating



In [ ]:
def selfplay(model, first_seed: int, games=1000):
    return run(games=games, first_seed=first_seed, cfg=Config(epsilon=0.0), 
               policy=model.predict, progress=False)

with open("../results/pickles/lgb5.pkl", "rb") as f:
    model = pickle.load(f)                                  # round 0: the original model

extras, rounds = [], []
for itr in range(4):
    start = time.time()
    test_ = evaluate(y[test], model.predict(W[test]), q[test], game[test], pos[test])
    ev = selfplay(model, 300000)                            # same evaluation games each round
    n = ev["game_pick_q_n"].sum()
    rounds.append({"round": itr,
                   "test regret": test_["regret"],
                   "self-play regret": ev["game_regret_sum"].sum() / n,
                   "guess mine chance": ev["game_pick_q_sum"].sum() / n,
                   "score": ev["game_score"].mean()})
    print(f"Round {itr} complete ({(time.time() - start):.2f}s)")
    start = time.time()
    if itr == 3:
        break
    extras.append(selfplay(model, 400000 + 10000 * itr))     # fresh seeds each round
    model = retrain_on(extras, lambda: GradientBoosted(k=5))
    with open(f"../results/pickles/lgb5_dagg_r{itr + 1}.pkl", "wb") as f:
        pickle.dump(model, f)
    print(f"lgb5_dagg_r{itr + 1} saved ({(time.time() - start):.2f}s)")


dagger_rounds = pd.DataFrame(rounds).set_index("round")
dagger_rounds["gap"] = dagger_rounds["self-play regret"] - dagger_rounds["test regret"]
dagger_rounds.round(3).to_csv("../results/dagg_rounds.csv")
print("\n >> dagg_rounds.csv saved")
display(dagger_rounds)